In [1]:
import sys
if '..' not in sys.path:
    sys.path.append("..")

In [2]:
from transformers import AutoTokenizer
from rl_retrieval.utils import all_facts_found
from rl_retrieval.feedback import GroundTruthFeedback
from rl_retrieval.llm_feedback import LLMJudge, ExactMatchFeedback, MutualInformationFeedback, StepwiseFactRelevanceFeedback

from rl_retrieval.retrieval_envs.qa_retrieval_env import QARetrievalEnv, SimpleEnvAdapter
from rl_retrieval.policy import RNDPolicy, OraclePolicy
from rl_retrieval.utils import all_facts_found
from dataloaders.localsets.musique import RetrievalMusique
from dataloaders.localsets.hotpotqa import RetrievalHotPotQA
from dataloaders.localsets.babilong import RetrievalBabilong

from dataloaders.globalset import PATHS
import numpy as np

In [3]:
"../" + PATHS['musique']

'../data_sources/musique'

In [5]:
seed = 42
split = 'eval'
num_samples = 5
#model_name = 'Qwen/Qwen-32B'
model_name = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
#tokenizer here are used only to estimate length of samples in datasets
dataset = RetrievalHotPotQA(
    path="../" + PATHS['hotpotqa'], #as we run this from notebooks/ subfloder
    tokenizer=tokenizer, length=-1,
    min_context_len=0, max_context_len=1e7,
    type='any', anno_type='any', split=split, seed=seed
)

dataset = SimpleEnvAdapter(dataset)
reward_model = None
termination_func = None

#fb_model = GroundTruthReward(per_fact_reward=0.05, completion_reward=1.)
#fb_model = LLM_Judge_Reward('Qwen/Qwen2.5-32B-Instruct', completion_reward=1.)
#fb_model = Exact_Match('Qwen/Qwen2.5-32B-Instruct', completion_reward=2.)
#fb_model = Information_Based_Reward('Qwen/Qwen2.5-32B-Instruct', completion_coeff=1.)
terminated = truncated = False
states = []
rewards = []
agent = OraclePolicy()

HotPotQA load:   0%|          | 0/7405 [00:00<?, ?it/s]

In [ ]:
dataset[7]

In [6]:
fb_model = StepwiseFactRelevanceFeedback(model_name, per_fact_reward=0.5)
#fb_model = LLMJudge(model_name, model_name)#ExactMatchFeedback(model_name)
env = QARetrievalEnv(fb_model, max_steps=3)
num_samples = 1

def compute_returns(rewards, gamma=1.):
    #Return or Reward-to-Go is computed with the following formula:
    # R_t = \sum^{T}_{k=t} r_k * \gamma^{k-t}
    returns = [None] * len(rewards)
    T = len(rewards)
    rtg = 0.
    for i in reversed(range(T)): 
        returns[i] = rewards[i] + gamma*rtg
        rtg = returns[i]
    return returns

for i in range(num_samples):
    step = 0
    terminated = truncated = False
    print(f"\n################## START EPISODE #{i} ####################")
    print(f'=========== Step #{step} ===========')
    ep_rewards = []
    obs, info = env.reset(dataset[i])
    print(f"Question: {obs['question']}?")
    print('Relevant chunks:')
    for j in info['sf_idx']:
        print(f"Chunk #{j}:\n {obs['chunks'][j]}\n")


    while not (terminated or truncated):
        # generally agent should avoid using info but these are just examples:
        chunk_idx = agent.act(obs, info)
        print(f'Agent selected chunks: {chunk_idx}')
        obs, reward, terminated, truncated, info = env.step(chunk_idx)
        ep_rewards.append(reward)
        print(f'reward - {reward}')
        print(f"last_action - {info['last_action']}")
        print(f"last_chunks - {obs['chunks'][info['last_action'][0]]}")
        step += 1
        print(f'Reward: {reward:.2f}, terminated: {terminated}, truncated: {truncated}')
        print(f"chunk mask: {obs['chunks_mask']}")
        print('Retrieved chunks:\n', "\n".join(obs['retrieved_chunks']), "\n")
        print(f'=========== Step #{step} ===========')

    returns = compute_returns(ep_rewards, gamma=0.99)
    print('Rewards for each step:', ep_rewards)
    print('Returns/RTG for each step:', returns)
    print(f"################## END EPISODE #{i} ####################")



[2025-05-14 17:27:19,732] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/griver/miniconda/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/griver/miniconda/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: /home/griver/miniconda/lib/libcufile.so: undefined reference to `dlvsym'
/home/griver/miniconda/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: /home/griver/miniconda/lib/libcufile.so: undefined reference to `dlopen'
/home/griver/miniconda/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: /home/griver/miniconda/lib/libcufile.so: undefined reference to `dlclose'
/home/griver/miniconda/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: /home/griver/miniconda/lib/libcufile.so: undefined reference to `dlerror'
/home/griver/miniconda/bin/../lib/gcc/x86_64-conda-linux-gnu/11

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]


################## START EPISODE #0 ####################
=========== Step #0 ===========
Question: VIVA Media AG changed it's name in 2004. What does their new acronym stand for?
Relevant chunks:
Chunk #5:
 VIVA Media VIVA Media GmbH (until 2004 "VIVA Media AG") is a music television network originating from Germany.  It was founded for broadcast of VIVA Germany as VIVA Media AG in 1993 and has been owned by their original concurrent Viacom, the parent company of MTV, since 2004.  Viva channels exist in some European countries; the first spin-offs were launched in Poland and Switzerland in 2000.

Chunk #7:
 Gesellschaft mit beschränkter Haftung A Gesellschaft mit beschränkter Haftung (] , abbreviated GmbH ] and also GesmbH in Austria) is a type of legal entity very common in Germany, Austria, Switzerland (where it is equivalent to a S.à r.l.) and Liechtenstein.  In the United States, the equivalent type of entity is the limited liability company (LLC).  The name of the GmbH form empha